# Energy and enstrophy fluxes
Plot one frame, several frames, or a pointwise frame average from `fluxes.csv`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'ns2d_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from ns2d_plotting import (available_frames, curves_for_frames, read_csv,
    repository_root, save_figure, select_frames, use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()
FLUXES_FILE = ROOT / 'output/fluxes.csv'
FIGURE_FILE = ROOT / 'figures/fluxes.pdf'

# 'single': one curve and exactly one selected frame.
# 'multiple': one curve for every selected frame.
# 'average': one pointwise average over all selected frames.
MODE = 'single'

# Negative frame indices count from the end. Set to None to use the range.
FRAMES = [-1]
FRAME_START = None
FRAME_STOP = None
FRAME_STRIDE = 1

# True uses the running segment-mean columns instead of instantaneous data.
USE_SEGMENT_MEAN = False
X_LOG_SCALE = True
# 'linear' shows signed flux directly; 'symlog' resolves small and large values.
Y_SCALE = 'linear'
USE_TEX = True
FONT_SIZE = 16

In [ ]:
use_plot_style(USE_TEX, FONT_SIZE)
table = read_csv(FLUXES_FILE)
frames = select_frames(available_frames(table), FRAMES, start=FRAME_START,
                       stop=FRAME_STOP, stride=FRAME_STRIDE)
energy_column = ('segment_mean_energy_flux' if USE_SEGMENT_MEAN
                 else 'energy_flux')
enstrophy_column = ('segment_mean_enstrophy_flux' if USE_SEGMENT_MEAN
                    else 'enstrophy_flux')
k, curves = curves_for_frames(table, frames,
                               [energy_column, enstrophy_column], MODE)
mask = np.isfinite(k) & ((k > 0.0) if X_LOG_SCALE else True)
print(f'Selected frames: {frames}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for curve in curves:
    energy = np.asarray(curve[energy_column])
    enstrophy = np.asarray(curve[enstrophy_column])
    axes[0].plot(k[mask], energy[mask], label=curve['label'])
    axes[1].plot(k[mask], enstrophy[mask], label=curve['label'])

for axis, ylabel in zip(axes, [r'$\Pi_E(k)$', r'$\Pi_Z(k)$']):
    axis.axhline(0.0, color='0.25', linewidth=0.8)
    axis.set_xlabel(r'$k$')
    axis.set_ylabel(ylabel)
    if X_LOG_SCALE:
        axis.set_xscale('log')
    axis.set_yscale(Y_SCALE)
    axis.grid(True, which='both', alpha=0.2)
    axis.legend()

saved = save_figure(fig, FIGURE_FILE)
print(f'Wrote {saved}')
plt.show()